In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Sat Mar 14 17:01:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install datasets tiktoken -q
print("Done!")

Done!


In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math
import os
import time
from datasets import load_dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

Using device: cuda
Number of GPUs: 2


In [4]:
# Smaller model — fits comfortably on single T4
VOCAB_SIZE    = 50257
BLOCK_SIZE    = 128     # ← reduced from 256
N_EMBD        = 384     # ← reduced from 512
N_HEAD        = 6       # ← reduced from 8
N_LAYER       = 6       # ← reduced from 8
DROPOUT       = 0.1

BATCH_SIZE    = 64      # single GPU batch
MAX_ITERS     = 50000
EVAL_INTERVAL = 500
EVAL_ITERS    = 50
LR            = 3e-4
LR_MIN        = 1e-5
ACCUM_STEPS   = 2       # effective batch = 128
GRAD_CLIP     = 1.0
SAVE_PATH     = "/kaggle/working/best_model.pt"

print("Config set!")
print(f"Estimated params: ~10-12M")
print(f"Single GPU mode")

Config set!
Estimated params: ~10-12M
Single GPU mode


In [5]:
import tiktoken

print("Loading tokenizer...")
enc = tiktoken.get_encoding("gpt2")

print("Loading TinyStories dataset (streaming)...")
dataset = load_dataset("roneneldan/TinyStories", split="train", streaming=True)

# We'll collect ~50M tokens — enough for a good 15M model run
# Takes about 3-4 minutes
MAX_TOKENS = 50_000_000
tokens = []

print("Tokenizing...")
start = time.time()

for example in dataset:
    text = example["text"]
    ids = enc.encode_ordinary(text)
    tokens.extend(ids)
    
    if len(tokens) >= MAX_TOKENS:
        break
    
    if len(tokens) % 5_000_000 == 0:
        print(f"  {len(tokens):,} tokens collected... ({time.time()-start:.0f}s)")

tokens = np.array(tokens, dtype=np.uint16)
print(f"\nDone! Total tokens: {len(tokens):,}")
print(f"Array size in memory: {tokens.nbytes / 1e6:.1f} MB")

Loading tokenizer...
Loading TinyStories dataset (streaming)...


Tokenizing...

Done! Total tokens: 50,000,035
Array size in memory: 100.0 MB


In [6]:
# Train/Val split (90/10)
n = len(tokens)
train_tokens = tokens[:int(n * 0.9)]
val_tokens   = tokens[int(n * 0.9):]

print(f"Train tokens: {len(train_tokens):,}")
print(f"Val tokens:   {len(val_tokens):,}")

class TokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size - 1

    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx   : idx+self.block_size], dtype=torch.long)
        y = torch.tensor(self.data[idx+1 : idx+self.block_size+1], dtype=torch.long)
        return x, y

train_dataset = TokenDataset(train_tokens, BLOCK_SIZE)
val_dataset   = TokenDataset(val_tokens,   BLOCK_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nTrain batches: {len(train_loader):,}")
print(f"Val batches:   {len(val_loader):,}")
print("Dataset ready!")

Train tokens: 45,000,031
Val tokens:   5,000,004

Train batches: 703,124
Val batches:   78,124
Dataset ready!


In [7]:
class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        assert N_EMBD % N_HEAD == 0
        self.n_head  = N_HEAD
        self.n_embd  = N_EMBD
        self.head_dim = N_EMBD // N_HEAD
        self.c_attn  = nn.Linear(N_EMBD, 3 * N_EMBD, bias=False)
        self.c_proj  = nn.Linear(N_EMBD, N_EMBD, bias=False)
        self.dropout = nn.Dropout(DROPOUT)
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE))
            .view(1, 1, BLOCK_SIZE, BLOCK_SIZE)
        )

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        scale = 1.0 / math.sqrt(self.head_dim)
        att = (q @ k.transpose(-2, -1)) * scale
        att = att.masked_fill(self.mask[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)
        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(out)

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_EMBD, 4 * N_EMBD, bias=False),
            nn.GELU(),
            nn.Linear(4 * N_EMBD, N_EMBD, bias=False),
            nn.Dropout(DROPOUT),
        )
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1  = nn.LayerNorm(N_EMBD)
        self.ln2  = nn.LayerNorm(N_EMBD)
        self.attn = CausalSelfAttention()
        self.ff   = FeedForward()
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(VOCAB_SIZE, N_EMBD)
        self.pos_emb   = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.drop      = nn.Dropout(DROPOUT)
        self.blocks    = nn.Sequential(*[TransformerBlock() for _ in range(N_LAYER)])
        self.ln_f      = nn.LayerNorm(N_EMBD)
        self.lm_head   = nn.Linear(N_EMBD, VOCAB_SIZE, bias=False)
        self.token_emb.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos    = torch.arange(0, T, device=idx.device)
        x      = self.token_emb(idx) + self.pos_emb(pos)
        x      = self.drop(x)
        x      = self.blocks(x)
        x      = self.ln_f(x)
        logits = self.lm_head(x)
        loss   = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=40):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -BLOCK_SIZE:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

# Single GPU only — no DataParallel
model = GPT().to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Device: {next(model.parameters()).device}")

# Quick VRAM check
torch.cuda.empty_cache()
allocated = torch.cuda.memory_allocated() / 1e9
print(f"VRAM used after model load: {allocated:.2f} GB")

Total parameters: 29,974,656
Device: cuda:0
VRAM used after model load: 0.12 GB


In [8]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    betas=(0.9, 0.95),
    weight_decay=0.1
)

# Cosine LR scheduler with warmup
# Warmup: LR ramps up for first 2000 steps, then cosine decays to LR_MIN
def get_lr(step):
    warmup_steps = 2000
    if step < warmup_steps:
        return LR * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / (MAX_ITERS - warmup_steps)
    cosine   = 0.5 * (1.0 + math.cos(math.pi * progress))
    return LR_MIN + (LR - LR_MIN) * cosine

# Enable mixed precision — cuts VRAM usage in half, 2x faster on T4
scaler = torch.cuda.amp.GradScaler()

print("Optimizer ready!")
print(f"Using mixed precision (fp16): True")
print(f"LR warmup steps: 2000")
print(f"LR schedule: Cosine decay {LR} → {LR_MIN}")

Optimizer ready!
Using mixed precision (fp16): True
LR warmup steps: 2000
LR schedule: Cosine decay 0.0003 → 1e-05


/tmp/ipykernel_200/4203223387.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [9]:
@torch.no_grad()
def estimate_val_loss():
    model.eval()
    losses = []
    val_iter = iter(val_loader)
    for _ in range(EVAL_ITERS):
        try:
            x, y = next(val_iter)
        except StopIteration:
            break
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast('cuda'):
            _, loss = model(x, y)
        losses.append(loss.mean().item())
    model.train()
    return np.mean(losses)


print("Starting training...")
print(f"Total iters: {MAX_ITERS:,} | Eval every: {EVAL_INTERVAL} | Batch: {BATCH_SIZE} | Accum: {ACCUM_STEPS}")
print("-" * 60)

train_iter    = iter(train_loader)
best_val_loss = float('inf')
step = 0
t0   = time.time()

while step < MAX_ITERS:
    # Gradient accumulation loop
    optimizer.zero_grad()
    accum_loss = 0.0

    for accum_step in range(ACCUM_STEPS):
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)

        x, y = x.to(device), y.to(device)

        with torch.amp.autocast('cuda'):
            _, loss = model(x, y)

        loss_mean = loss.mean() / ACCUM_STEPS   # normalize by accum steps
        scaler.scale(loss_mean).backward()
        accum_loss += loss_mean.item()

    # Update LR
    lr = get_lr(step)
    for g in optimizer.param_groups:
        g['lr'] = lr

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()

    step += 1

    if step % 100 == 0:
        t1  = time.time()
        dt  = (t1 - t0) / 100
        eta = dt * (MAX_ITERS - step) / 3600
        print(f"step {step:5d} | loss {accum_loss:.4f} | lr {lr:.2e} | {dt*1000:.0f}ms/step | ETA {eta:.1f}h")
        t0  = time.time()

    if step % EVAL_INTERVAL == 0:
        val_loss = estimate_val_loss()
        print(f"\n{'='*60}")
        print(f"EVAL @ step {step} | train_loss {accum_loss:.4f} | val_loss {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            raw = model.module if hasattr(model, 'module') else model
            torch.save({
                'step':      step,
                'model':     raw.state_dict(),
                'optimizer': optimizer.state_dict(),
                'val_loss':  val_loss,
                'config': {
                    'n_embd': N_EMBD, 'n_head': N_HEAD,
                    'n_layer': N_LAYER, 'block_size': BLOCK_SIZE,
                    'vocab_size': VOCAB_SIZE
                }
            }, SAVE_PATH)
            print(f"✅ New best model saved! val_loss = {val_loss:.4f}")
        print(f"{'='*60}\n")

print("\nTraining complete!")

Starting training...
Total iters: 50,000 | Eval every: 500 | Batch: 64 | Accum: 2
------------------------------------------------------------
step   100 | loss 9.5949 | lr 1.50e-05 | 408ms/step | ETA 5.6h
step   200 | loss 8.0488 | lr 3.00e-05 | 410ms/step | ETA 5.7h
step   300 | loss 6.2507 | lr 4.50e-05 | 412ms/step | ETA 5.7h
step   400 | loss 5.0873 | lr 6.00e-05 | 408ms/step | ETA 5.6h
step   500 | loss 4.4642 | lr 7.50e-05 | 408ms/step | ETA 5.6h

EVAL @ step 500 | train_loss 4.4642 | val_loss 4.7156
✅ New best model saved! val_loss = 4.7156

step   600 | loss 4.1246 | lr 9.00e-05 | 457ms/step | ETA 6.3h
step   700 | loss 3.8229 | lr 1.05e-04 | 407ms/step | ETA 5.6h
step   800 | loss 3.6873 | lr 1.20e-04 | 406ms/step | ETA 5.6h
step   900 | loss 3.4953 | lr 1.35e-04 | 406ms/step | ETA 5.5h
step  1000 | loss 3.3711 | lr 1.50e-04 | 406ms/step | ETA 5.5h

EVAL @ step 1000 | train_loss 3.3711 | val_loss 3.6505
✅ New best model saved! val_loss = 3.6505

step  1100 | loss 3.2467 | lr 

## training completed now finetunung steps


In [2]:
import os
# Find where kaggle put your uploaded file
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/dhruvkumar11/trained-model/best_model.pt


In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math
import os
import time
import json
import tiktoken
from datasets import load_dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Same config as pretraining — must match exactly
VOCAB_SIZE  = 50257
BLOCK_SIZE  = 128
N_EMBD      = 384
N_HEAD      = 6
N_LAYER     = 6
DROPOUT     = 0.1

CHECKPOINT_PATH = "/kaggle/input/datasets/dhruvkumar11/trained-model/best_model.pt"
SFT_SAVE_PATH   = "/kaggle/working/sft_model.pt"

print("Config ready!")

Device: cuda
GPU: Tesla T4
Config ready!


In [2]:
class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        assert N_EMBD % N_HEAD == 0
        self.n_head   = N_HEAD
        self.n_embd   = N_EMBD
        self.head_dim = N_EMBD // N_HEAD
        self.c_attn   = nn.Linear(N_EMBD, 3 * N_EMBD, bias=False)
        self.c_proj   = nn.Linear(N_EMBD, N_EMBD,     bias=False)
        self.dropout  = nn.Dropout(DROPOUT)
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE))
            .view(1, 1, BLOCK_SIZE, BLOCK_SIZE)
        )

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        scale = 1.0 / math.sqrt(self.head_dim)
        att = (q @ k.transpose(-2, -1)) * scale
        att = att.masked_fill(self.mask[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)
        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(out)

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_EMBD, 4 * N_EMBD, bias=False),
            nn.GELU(),
            nn.Linear(4 * N_EMBD, N_EMBD, bias=False),
            nn.Dropout(DROPOUT),
        )
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1  = nn.LayerNorm(N_EMBD)
        self.ln2  = nn.LayerNorm(N_EMBD)
        self.attn = CausalSelfAttention()
        self.ff   = FeedForward()
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(VOCAB_SIZE, N_EMBD)
        self.pos_emb   = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.drop      = nn.Dropout(DROPOUT)
        self.blocks    = nn.Sequential(*[TransformerBlock() for _ in range(N_LAYER)])
        self.ln_f      = nn.LayerNorm(N_EMBD)
        self.lm_head   = nn.Linear(N_EMBD, VOCAB_SIZE, bias=False)
        self.token_emb.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos    = torch.arange(0, T, device=idx.device)
        x      = self.token_emb(idx) + self.pos_emb(pos)
        x      = self.drop(x)
        x      = self.blocks(x)
        x      = self.ln_f(x)
        logits = self.lm_head(x)
        loss   = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=40):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -BLOCK_SIZE:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

print("Model architecture defined!")

Model architecture defined!


In [4]:
enc = tiktoken.get_encoding("gpt2")

# Fix for PyTorch 2.6+
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model = GPT().to(device)
model.load_state_dict(checkpoint['model'])
model.eval()

print(f"✅ Loaded pretrained model!")
print(f"   Step: {checkpoint['step']:,}")
print(f"   Val loss: {checkpoint['val_loss']:.4f}")

print("\n" + "="*60)
print("BEFORE FINETUNING — Raw pretrained output:")
print("="*60)

def generate(prompt, max_tokens=150, temperature=0.8, top_k=40):
    tokens = enc.encode(prompt)
    x = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
    with torch.no_grad():
        out = model.generate(x, max_new_tokens=max_tokens,
                           temperature=temperature, top_k=top_k)
    return enc.decode(out[0].tolist())

test_prompts = [
    "Once upon a time there was a little girl",
    "The dog ran fast because",
    "Tom and his friend went to the park and",
]

for p in test_prompts:
    print(f"\nPrompt: {p}")
    print(generate(p))
    print()

✅ Loaded pretrained model!
   Step: 49,000
   Val loss: 2.1158

BEFORE FINETUNING — Raw pretrained output:

Prompt: Once upon a time there was a little girl
Once upon a time there was a little girl named Jane. She was very excited for her birthday. In the morning they were both ready for the party. Jane wanted to go to the party. She asked her mom to help her.

"Mom, can I go to the party now?" Jane asked.

"Yes, sweetie," her mom said. "But first, we need to get ready for the party."

Jane was so excited. She put on her special dress and ran to the party.

When she arrived, she saw that there were lots of kids playing and laughing. Then she saw all of the kids playing and having fun. She joined them and played too.

Jane had a great time and was so happy she could go to


Prompt: The dog ran fast because
The dog ran fast because it was a good run. It came near a tree and saw the ball. It wanted the ball. It barked to the tree and tried to get the ball.

The dog did not want the ball. 

In [5]:
print("Loading Alpaca dataset...")
alpaca = load_dataset("tatsu-lab/alpaca", split="train")
print(f"Total examples: {len(alpaca)}")
print(f"\nSample example:")
print(alpaca[0])

Loading Alpaca dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Total examples: 52002

Sample example:
{'instruction': 'Give three tips for staying healthy.', 'input': '', 'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.', 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'}


In [6]:
# This is the exact prompt format we'll train on
# Same format used by Stanford Alpaca / LLaMA finetuning
PROMPT_TEMPLATE = """### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

PROMPT_NO_INPUT = """### Instruction:
{instruction}

### Response:
{output}"""

def format_example(example):
    if example["input"].strip():
        return PROMPT_TEMPLATE.format(
            instruction=example["instruction"],
            input=example["input"],
            output=example["output"]
        )
    else:
        return PROMPT_NO_INPUT.format(
            instruction=example["instruction"],
            output=example["output"]
        )

# Format all examples
print("Formatting dataset...")
formatted = [format_example(ex) for ex in alpaca]

# Check average length
lengths = [len(enc.encode(t)) for t in formatted[:500]]
print(f"Sample formatted example:\n{formatted[0]}")
print(f"\nAvg token length: {np.mean(lengths):.0f}")
print(f"Max token length: {max(lengths):.0f}")
print(f"Total examples: {len(formatted)}")

Formatting dataset...
Sample formatted example:
### Instruction:
Give three tips for staying healthy.

### Response:
1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. 
2. Exercise regularly to keep your body active and strong. 
3. Get enough sleep and maintain a consistent sleep schedule.

Avg token length: 81
Max token length: 407
Total examples: 52002


In [7]:
class AlpacaDataset(Dataset):
    def __init__(self, examples, enc, block_size):
        self.block_size = block_size
        self.data = []

        print("Tokenizing Alpaca dataset...")
        skipped = 0
        for text in examples:
            tokens = enc.encode(text)
            # Skip examples longer than block_size
            if len(tokens) > block_size - 1:
                skipped += 1
                continue
            # Pad to block_size with -1 (we'll mask these in loss)
            self.data.append(tokens)

        print(f"Valid examples: {len(self.data)}")
        print(f"Skipped (too long): {skipped}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens = self.data[idx]
        # Pad sequence to block_size
        padded = tokens + [0] * (self.block_size - len(tokens))
        x = torch.tensor(padded[:-1], dtype=torch.long)
        y = torch.tensor(padded[1:],  dtype=torch.long)
        return x, y

# Split 90/10
split = int(len(formatted) * 0.9)
train_data = formatted[:split]
val_data   = formatted[split:]

train_sft = AlpacaDataset(train_data, enc, BLOCK_SIZE)
val_sft   = AlpacaDataset(val_data,   enc, BLOCK_SIZE)

train_sft_loader = DataLoader(train_sft, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_sft_loader   = DataLoader(val_sft,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nTrain batches: {len(train_sft_loader)}")
print(f"Val batches:   {len(val_sft_loader)}")
print("SFT Dataset ready!")

Tokenizing Alpaca dataset...
Valid examples: 38197
Skipped (too long): 8604
Tokenizing Alpaca dataset...
Valid examples: 4202
Skipped (too long): 999

Train batches: 1194
Val batches:   132
SFT Dataset ready!


In [8]:
# SFT specific hyperparams
# Lower LR than pretraining — we're nudging, not retraining
SFT_LR       = 5e-5      # 6x smaller than pretraining LR
SFT_LR_MIN   = 1e-6
SFT_EPOCHS   = 3         # 3 full passes over Alpaca
GRAD_CLIP    = 1.0
WARMUP_STEPS = 100

# Reinitialize optimizer with lower LR
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=SFT_LR,
    betas=(0.9, 0.95),
    weight_decay=0.01    # lighter decay for finetuning
)

scaler = torch.cuda.amp.GradScaler()

total_steps = SFT_EPOCHS * len(train_sft_loader)

def get_sft_lr(step):
    if step < WARMUP_STEPS:
        return SFT_LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / (total_steps - WARMUP_STEPS)
    cosine   = 0.5 * (1.0 + math.cos(math.pi * progress))
    return SFT_LR_MIN + (SFT_LR - SFT_LR_MIN) * cosine

print(f"Total SFT steps: {total_steps:,}")
print(f"Epochs: {SFT_EPOCHS}")
print(f"LR: {SFT_LR} → {SFT_LR_MIN} (cosine)")
print("Ready!")

Total SFT steps: 3,582
Epochs: 3
LR: 5e-05 → 1e-06 (cosine)
Ready!


/tmp/ipykernel_55/2865736129.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [9]:
@torch.no_grad()
def sft_val_loss():
    model.eval()
    losses = []
    for i, (x, y) in enumerate(val_sft_loader):
        if i >= 50: break
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast('cuda'):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return np.mean(losses)


model.train()
best_sft_loss = float('inf')
global_step   = 0
t0 = time.time()

print("="*60)
print("Starting SFT Finetuning...")
print("="*60)

for epoch in range(SFT_EPOCHS):
    epoch_losses = []

    for batch_idx, (x, y) in enumerate(train_sft_loader):
        x, y = x.to(device), y.to(device)

        # Update LR
        lr = get_sft_lr(global_step)
        for g in optimizer.param_groups:
            g['lr'] = lr

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            _, loss = model(x, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        epoch_losses.append(loss.item())
        global_step += 1

        # Log every 100 steps
        if global_step % 100 == 0:
            t1  = time.time()
            dt  = (t1 - t0) / 100
            eta = dt * (total_steps - global_step) / 3600
            print(f"epoch {epoch+1} | step {global_step:4d}/{total_steps} | "
                  f"loss {loss.item():.4f} | lr {lr:.2e} | "
                  f"{dt*1000:.0f}ms/step | ETA {eta:.2f}h")
            t0 = time.time()

    # End of epoch — full eval
    val_loss   = sft_val_loss()
    epoch_loss = np.mean(epoch_losses)
    print(f"\n{'='*60}")
    print(f"EPOCH {epoch+1} COMPLETE")
    print(f"Train loss: {epoch_loss:.4f} | Val loss: {val_loss:.4f}")

    if val_loss < best_sft_loss:
        best_sft_loss = val_loss
        raw = model.module if hasattr(model, 'module') else model
        torch.save({
            'epoch':     epoch + 1,
            'model':     raw.state_dict(),
            'optimizer': optimizer.state_dict(),
            'val_loss':  val_loss,
            'config': {
                'n_embd': N_EMBD, 'n_head': N_HEAD,
                'n_layer': N_LAYER, 'block_size': BLOCK_SIZE,
                'vocab_size': VOCAB_SIZE
            }
        }, SFT_SAVE_PATH)
        print(f"✅ Best SFT model saved! val_loss = {val_loss:.4f}")
    print(f"{'='*60}\n")

print("SFT Finetuning Complete!")

Starting SFT Finetuning...
epoch 1 | step  100/3582 | loss 2.8182 | lr 5.00e-05 | 117ms/step | ETA 0.11h
epoch 1 | step  200/3582 | loss 2.5873 | lr 4.99e-05 | 113ms/step | ETA 0.11h
epoch 1 | step  300/3582 | loss 2.7880 | lr 4.96e-05 | 114ms/step | ETA 0.10h
epoch 1 | step  400/3582 | loss 2.2750 | lr 4.91e-05 | 116ms/step | ETA 0.10h
epoch 1 | step  500/3582 | loss 2.0221 | lr 4.84e-05 | 118ms/step | ETA 0.10h
epoch 1 | step  600/3582 | loss 1.9348 | lr 4.76e-05 | 118ms/step | ETA 0.10h
epoch 1 | step  700/3582 | loss 2.0384 | lr 4.65e-05 | 117ms/step | ETA 0.09h
epoch 1 | step  800/3582 | loss 2.0362 | lr 4.53e-05 | 115ms/step | ETA 0.09h
epoch 1 | step  900/3582 | loss 2.2058 | lr 4.39e-05 | 115ms/step | ETA 0.09h
epoch 1 | step 1000/3582 | loss 2.1405 | lr 4.24e-05 | 115ms/step | ETA 0.08h
epoch 1 | step 1100/3582 | loss 1.8831 | lr 4.07e-05 | 115ms/step | ETA 0.08h

EPOCH 1 COMPLETE
Train loss: 2.3432 | Val loss: 2.0526
✅ Best SFT model saved! val_loss = 2.0526

epoch 2 | step 1

In [11]:
@torch.no_grad()
def generate_no_repeat(prompt_text, max_tokens=150, temperature=0.85, top_k=40, rep_penalty=1.3):
    tokens = enc.encode(prompt_text)
    x = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
    
    generated = list(tokens)
    
    for _ in range(max_tokens):
        inp = torch.tensor([generated[-BLOCK_SIZE:]], dtype=torch.long, device=device)
        with torch.amp.autocast('cuda'):
            logits, _ = model(inp)
        
        logits = logits[0, -1, :]  # last token logits
        
        # Apply repetition penalty
        for token_id in set(generated[-64:]):  # penalize recent tokens
            logits[token_id] /= rep_penalty
        
        # Temperature + top-k
        logits = logits / temperature
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[-1]] = float('-inf')
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1).item()
        
        generated.append(next_token)
        
        # Stop at natural ending
        decoded = enc.decode([next_token])
        if decoded in ['\n\n', '<|endoftext|>']:
            break
    
    return enc.decode(generated[len(tokens):])


def chat(instruction, input_text="", max_tokens=150):
    if input_text.strip():
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    return generate_no_repeat(prompt, max_tokens=max_tokens)


# Test again
tests = [
    ("Write a short story about a brave cat.", ""),
    ("Explain what the sun is in simple words.", ""),
    ("What is 2 + 2? Explain your answer.", ""),
    ("Summarize this text.", "The dog went to the park and played with a ball."),
]

print("="*60)
print("FIXED — With Repetition Penalty:")
print("="*60)

for instruction, inp in tests:
    print(f"\nInstruction: {instruction}")
    if inp:
        print(f"Input: {inp}")
    print(f"Response: {chat(instruction, inp)}")
    print("-"*40)

FIXED — With Repetition Penalty:

Instruction: Write a short story about a brave cat.
Response: "I'm so proud of my intelligent ways."!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
----------------------------------------

Instruction: Explain what the sun is in simple words.
Response: The sun works as it stands on a tree, growing on its roots of leaves and branches like a breeze. It also becomes anton-free andful night sky. As you walk around, the heat makes everything feel warm and happy.!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
----------------------------------------

Instruction: What is 2 + 2? Explain your answer.
Response: The five adject2 of 10 are 0.!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
------------------------------------

In [12]:
@torch.no_grad()
def chat_fixed(instruction, input_text="", max_tokens=150, temperature=0.9, top_k=50):
    if input_text.strip():
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    tokens = enc.encode(prompt)
    generated = list(tokens)
    prev_token = None
    repeat_count = 0

    for _ in range(max_tokens):
        inp = torch.tensor([generated[-BLOCK_SIZE:]], dtype=torch.long, device=device)
        logits, _ = model(inp)
        logits = logits[0, -1, :]

        # Hard block: zero out any token seen 3+ times in last 20 tokens
        recent = generated[-20:]
        for tid in set(recent):
            if recent.count(tid) >= 3:
                logits[tid] = float('-inf')

        # Temperature + top-k
        logits = logits / temperature
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[-1]] = float('-inf')
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1).item()

        # Stop if repeating same token
        if next_token == prev_token:
            repeat_count += 1
            if repeat_count >= 3:
                break
        else:
            repeat_count = 0
        prev_token = next_token

        generated.append(next_token)

        # Stop at end of text or double newline
        text_so_far = enc.decode(generated[len(tokens):])
        if '<|endoftext|>' in text_so_far or text_so_far.endswith('\n\n'):
            break

    response = enc.decode(generated[len(tokens):])
    # Clean up
    response = response.replace('<|endoftext|>', '').strip()
    # Cut at next ### if model starts a new instruction
    if '###' in response:
        response = response.split('###')[0].strip()
    return response


# Test
tests = [
    ("Write a short story about a brave cat.", ""),
    ("Explain what the sun is in simple words.", ""),
    ("What is 2 + 2?", ""),
    ("Summarize this text.", "The dog went to the park and played with a ball."),
    ("What is the capital of France?", ""),
]

print("="*60)
for instruction, inp in tests:
    print(f"\nInstruction: {instruction}")
    if inp: print(f"Input: {inp}")
    print(f"Response: {chat_fixed(instruction, inp)}")
    print("-"*40)


Instruction: Write a short story about a brave cat.
Response: In this story, he encountered some bad problems that were not brave towards them, but he was determined to try to protect them; he must be careful and never give up until a strong mission comes. He eventually completed a mission, and the mouse was caught of the cat's safety.!!!!" said the cat. and "If you ever need help, go out and find a way to protect your family from dangerous problems."!!!!". and "Froar course, you are brave and smart. You should also use the ability to do what you're afraid of, even if it is hard work.!!!!" #F. # #1 = Additionally |2+5 +2] 
[noinput
----------------------------------------

Instruction: Explain what the sun is in simple words.
Response: The sun is in the sky.!!!.!"   
No sentence.
----------------------------------------

Instruction: What is 2 + 2?
Response: 2°3 + 4!!! #2> (5) 5) (2) 710 = 44 and 6.!!! - 9 + 5^32.21 - 4) and 3 (3
58!!! - 6 + 2)
9 10 - 8/3/4)
--------------------------